# Tikhonov-regularised downward continuation (Tikhonov et al., 1968)

Classical baseline, no training. The downward-continuation filter is $e^{kz} / (1 + \alpha k^2 e^{kz})$,
with α chosen in one of two ways:

* **TRDC**: α at the corner (maximum curvature) of the L-curve, using no ground truth.
* **Oracle TRDC**: α minimising the RMS error to the ground truth, i.e. an upper bound for TRDC.

Runs on CPU.

**Evaluation protocol (all notebooks).** The 10 field test patches (32 × 32 nodes, 400 m spacing) are
scaled to [-1, 1] per patch. PSNR, SSIM, relative error (RE) and RMSE are computed on the central 24 × 24
window and reported as mean ± standard deviation (ddof = 0) over the 10 patches. Networks run in full FP32
with deterministic cuDNN (set when `gravdc` is imported, see `gravdc/precision.py`).

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))  # make the gravdc package importable

import numpy as np
import torch

from gravdc import evaluate, format_summary, load_model, load_test_set, save_result, summarize
from gravdc.plotting import show_reconstruction
from gravdc.table3 import compare

DATA_DIR = "../data/test"
RESULTS_DIR = "../results"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)

device: cuda | torch 2.9.0+cu128


## Data
Observation: the multiscale operator (40 steps of 20 m = 800 m), as for TTSIDC.

In [2]:
from gravdc import MultiscaleOperator
from gravdc import classical

names, x = load_test_set(DATA_DIR)
A = MultiscaleOperator()  # 40 x 20 m = 800 m
y = A(x)
print(len(names), "patches:", names)
TRDC = dict(perex=(4, 4), z=800, dx=400, dy=400)

10 patches: ['data14.mat', 'data16.mat', 'data18.mat', 'data20.mat', 'data22.mat', 'data24.mat', 'data26.mat', 'data28.mat', 'data30.mat', 'data32.mat']


## Inference

In [3]:
rec_lcurve, rec_oracle = [], []
for name, f, truth in zip(names, y[:, 0].numpy(), x[:, 0].numpy()):
    alpha_l = classical.lcurve_alpha(f, n=200, alpha_start=1e-4, ratio=1.1, **TRDC)
    alpha_o = classical.oracle_alpha(f, truth, n=2000, alpha_start=0.1, ratio=1.01, **TRDC)
    rec_lcurve.append(classical.tr_down_continue(f, alpha_l, **TRDC))
    rec_oracle.append(classical.tr_down_continue(f, alpha_o, **TRDC))
    print(f"{name:12s} alpha (L-curve) = {alpha_l:9.4f}   alpha (oracle) = {alpha_o:9.4f}")
rec_lcurve = np.stack(rec_lcurve)[:, None]
rec_oracle = np.stack(rec_oracle)[:, None]

data14.mat   alpha (L-curve) =  346.7722   alpha (oracle) =   56.5868
data16.mat   alpha (L-curve) =  381.4494   alpha (oracle) =   61.2754
data18.mat   alpha (L-curve) =  195.7439   alpha (oracle) =   46.8391
data20.mat   alpha (L-curve) =  215.3182   alpha (oracle) =   51.2273
data22.mat   alpha (L-curve) =  346.7722   alpha (oracle) =   60.6687
data24.mat   alpha (L-curve) =  419.5943   alpha (oracle) =   60.6687
data26.mat   alpha (L-curve) =  286.5886   alpha (oracle) =   58.8844
data28.mat   alpha (L-curve) =  195.7439   alpha (oracle) =   51.7396
data30.mat   alpha (L-curve) =  133.6957   alpha (oracle) =   45.0115
data32.mat   alpha (L-curve) =  346.7722   alpha (oracle) =  401.8146


## Metrics

In [4]:
for row, rec in [("Oracle TRDC", rec_oracle), ("TRDC", rec_lcurve)]:
    per_image = evaluate(x, rec)
    compare(row, summarize(per_image))
    save_result(row, per_image, names, RESULTS_DIR)

Oracle TRDC
  metric             this run              Table 3
  PSNR       40.9487 ± 2.7437     40.9487 ± 2.7437  ✓
  SSIM        0.9910 ± 0.0051      0.9910 ± 0.0051  ✓
  RE          0.0455 ± 0.0125      0.0455 ± 0.0125  ✓
  RMSE        0.0160 ± 0.0055      0.0160 ± 0.0055  ✓
TRDC
  metric             this run              Table 3
  PSNR       37.7190 ± 2.1283     37.7190 ± 2.1283  ✓
  SSIM        0.9862 ± 0.0084      0.9862 ± 0.0084  ✓
  RE          0.0656 ± 0.0176      0.0656 ± 0.0176  ✓
  RMSE        0.0231 ± 0.0080      0.0231 ± 0.0080  ✓


In [5]:
# Figures of the field data are off by default (the executed notebooks are kept small and text-only).
# Set SHOW_FIGURE = True to look at a reconstruction.
SHOW_FIGURE = False
if SHOW_FIGURE:
    k = 0
    show_reconstruction(x[k], y[k], {"TRDC": rec_lcurve[k], "Oracle TRDC": rec_oracle[k]}, title=names[k])